# Description
...

In [4]:
import pandas as pd
import plotly.express as px
from tqdm import tqdm
import numpy as np
import re
from scipy.spatial.distance import jensenshannon

In [38]:
# Read in necessary tables

courses = pd.read_csv("data/courses.csv", sep=";", encoding="utf-8", index_col=0)
writers = pd.read_csv("data/writers.csv", sep=";", encoding="utf-8", index_col=0)


In [8]:
courses.head()

,ID,title,semester,semester_start,inhalt_count,university,country,NER_GNDs
0,wien0001,Einf.PS: Neuere dt. Literaturwissenschaft I,wise00-01,2000-09,0,wien,Österreich,NaN
1,wien0002,Seminar: Neuere deutsche Literatur,wise00-01,2000-09,0,wien,Österreich,NaN
2,wien0003,SE: Neuere dt. Literatur-Poetik u.Transgression,wise00-01,2000-09,0,wien,Österreich,NaN
3,wien0004,EPS: Neuere dt. Literaturwissenschaft II,wise00-01,2000-09,0,wien,Österreich,NaN
4,wien0005,Proseminar: Neuere deutsche Literaten,wise00-01,2000-09,0,wien,Österreich,NaN


In [11]:
writers.head()

,GND-ID,occupation_GND,birth_GND,gender,death_GND,country_GND,country_norm
0,118544918,Schriftsteller; Lyriker; Publizist; Nonne,1805-06-22,Weiblich,1880-01-12,Frankreich; Deutschland,D
1,118577166,Nobelpreisträger; Musiker; Intellektueller; Sc...,1875-06-06,Männlich,1955-08-12,Deutschland; Japan; Europa; USA,"D,S"
2,118570366,Schriftsteller; Offizier; Archäologe; Geheimagent,1888-08-16,Männlich,1935-05-19,Großbritannien,NaN
3,133347931,Schriftsteller; Architekt,1974,Männlich,NaN,Deutschland,D
4,118552465,Komponist; Schriftsteller; Kapellmeister; Zeic...,1776-01-24,Männlich,1822-06-25,Deutschland,D


In [39]:
# Create list of the columns needed
#future_columns = ["total", "total_G", "total_A", "total_S", "rel_G", "rel_A", "rel_S"]
future_columns = []
unis = set(courses.university.to_list())
for uni in unis:
    future_columns.append("total_"+uni)
    #future_columns.append("rel_"+uni)

future_columns


['total_graz',
 'total_mainz',
 'total_chemnitz',
 'total_basel',
 'total_stuttgart',
 'total_marburg',
 'total_halle',
 'total_wien',
 'total_erfurt']

In [40]:
# Create dictionary where counts will be stored
gnd_ids = writers["GND-ID"].to_list()
counts = {col: {num: 0 for num in gnd_ids} for col in future_columns}

In [41]:
# Count number of course descriptions per uni
content_counts = {uni: 0 for uni in unis}
for uni in content_counts.keys():
    
    uni_courses = courses[courses["university"] == uni]
    contentcount = len(uni_courses[uni_courses["inhalt_count"]!=0])
    content_counts[uni] = contentcount
content_counts

{'graz': 1400,
 'mainz': 540,
 'chemnitz': 131,
 'basel': 623,
 'stuttgart': 681,
 'marburg': 436,
 'halle': 1291,
 'wien': 704,
 'erfurt': 321}

In [42]:
# compute counts

for i in range(len(courses)):
    ids = courses.NER_GNDs.loc[i]
    uni = courses["university"].loc[i]
    country = courses["country"].loc[i]
    
    if not pd.isna(ids):
        ids = re.sub("\[|\]|\'", "", ids).split(", ")
        for id in ids: 
            if id in gnd_ids: # konnte Person identifiziert werden und ist Autor:in?
                counts["total_"+uni][id] += 1
               


In [43]:
df = pd.DataFrame(counts)

In [44]:
for uni in unis:
    df["rel_"+uni] = df["total_"+uni]/content_counts[uni]
df

,total_graz,total_mainz,total_chemnitz,total_basel,total_stuttgart,total_marburg,total_halle,total_wien,total_erfurt,rel_graz,rel_mainz,rel_chemnitz,rel_basel,rel_stuttgart,rel_marburg,rel_halle,rel_wien,rel_erfurt
118544918,0,0,0,1,0,0,0,0,0,0.000000,0.0,0.0,0.001605,0.0,0.0,0.000000,0.000000,0.000000
118577166,0,0,0,0,0,0,0,0,0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000
118570366,0,0,0,1,0,0,0,0,2,0.000000,0.0,0.0,0.001605,0.0,0.0,0.000000,0.000000,0.006231
133347931,0,0,0,0,0,0,2,3,0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.001549,0.004261,0.000000
118552465,0,0,0,0,0,0,2,0,0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.001549,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138089981,1,0,0,0,0,0,0,0,0,0.000714,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000
118780247,0,0,0,2,0,0,0,0,0,0.000000,0.0,0.0,0.003210,0.0,0.0,0.000000,0.000000,0.000000
14009931X,0,0,0,0,0,0,1,0,0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000775,0.000000,0.000000
1128925060,1,0,0,0,0,0,0,0,0,0.000714,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000


In [49]:
df["total_A"] = df["total_graz"] + df["total_wien"]
df["total_S"] = df["total_basel"]
df["total_G"] = df["total_mainz"] + df["total_chemnitz"] + df["total_stuttgart"] + \
                df["total_marburg"] + df["total_halle"] + df["total_erfurt"]

df["rel_G"] = (df["rel_mainz"] + df["rel_chemnitz"] + df["rel_stuttgart"] + \
                df["rel_marburg"] + df["rel_halle"] + df["rel_erfurt"] )/6
df["rel_A"] = (df["rel_graz"] + df["rel_wien"])/2
df["rel_S"] = df["rel_basel"]
df

,total_graz,total_mainz,total_chemnitz,total_basel,total_stuttgart,total_marburg,total_halle,total_wien,total_erfurt,rel_graz,...,rel_marburg,rel_halle,rel_wien,rel_erfurt,total_A,total_S,total_G,rel_G,rel_A,rel_S
118544918,0,0,0,1,0,0,0,0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0,1,0,0.000000,0.000000,0.001605
118577166,0,0,0,0,0,0,0,0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.000000
118570366,0,0,0,1,0,0,0,0,2,0.000000,...,0.0,0.000000,0.000000,0.006231,0,1,2,0.001038,0.000000,0.001605
133347931,0,0,0,0,0,0,2,3,0,0.000000,...,0.0,0.001549,0.004261,0.000000,3,0,2,0.000258,0.002131,0.000000
118552465,0,0,0,0,0,0,2,0,0,0.000000,...,0.0,0.001549,0.000000,0.000000,0,0,2,0.000258,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138089981,1,0,0,0,0,0,0,0,0,0.000714,...,0.0,0.000000,0.000000,0.000000,1,0,0,0.000000,0.000357,0.000000
118780247,0,0,0,2,0,0,0,0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0,2,0,0.000000,0.000000,0.003210
14009931X,0,0,0,0,0,0,1,0,0,0.000000,...,0.0,0.000775,0.000000,0.000000,0,0,1,0.000129,0.000000,0.000000
1128925060,1,0,0,0,0,0,0,0,0,0.000714,...,0.0,0.000000,0.000000,0.000000,1,0,0,0.000000,0.000357,0.000000


In [50]:
df.to_csv("data/writer_counts.csv", sep=";", encoding="utf-8")
